# PYPOWER AC-OPF on the PJM 5-Bus Network (Reference Solution)
AC optimal power flow on the PJM 5-bus case (pglib case5_pjm) solved with PYPOWER. This notebook is the reference solution for the Pyomo AC-OPF notebook: it provides an independent dispatch, cost, bus LMPs and line shadow prices to compare against.

**Contents**
* [Setup](#Setup)
* [Case Data](#Case-Data)
* [Run AC-OPF](#Run-AC-OPF)
* [Results](#Results)
* [Reference Values for the Pyomo Notebook](#Reference-Values-for-the-Pyomo-Notebook)


## Setup
PYPOWER is a Python port of MATPOWER. `runopf` builds the AC optimal power flow and solves it with PIPS, an interior-point solver. Install once with `pip install pypower`.

In [1]:
!pip install pypower -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.6/347.6 kB 9.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from pypower.api import runopf, ppoption
from pypower.idx_bus import BUS_I, VM, VA, LAM_P, LAM_Q            # bus: id, voltage magnitude (p.u.), angle (deg), LMP ($/MWh), reactive LMP ($/MVAr.h)
from pypower.idx_gen import GEN_BUS, PG, QG, PMAX, QMAX, QMIN      # gen: bus, active/reactive output, limits
from pypower.idx_brch import F_BUS, T_BUS, RATE_A, PF, QF, PT, QT, MU_SF, MU_ST   # branch: ends, limit, flows at both ends, multipliers

pd.options.display.float_format = "{:,.4f}".format

## Case Data
Same network as the Pyomo notebook: 5 buses, 5 generators and 6 lines, with Branch 6 (Bus 4 - Bus 5, 240 MVA) as the tightest line. PYPOWER takes the case as a dictionary of arrays in MATPOWER format. Power is in MW, and `gencost` uses the polynomial model (`2`) with 3 coefficients `c2, c1, c0`. With `c2 = 0` every cost is linear in \$/MWh.

AC-OPF also uses the reactive data: `Qd` on the buses, `Qmin/Qmax` on the generators, the voltage bounds `Vmin/Vmax` (0.9 to 1.1 p.u.) and the line resistance `r` and charging `b`. A branch rating (`rateA`) is an **apparent power** limit in MVA.

In [3]:
net = {
    "baseMVA": 100.0,

    # areas: area, reference bus
    "areas": np.array([[1, 4]]),

    # bus: bus_i, type, Pd, Qd, Gs, Bs, area, Vm, Va, baseKV, zone, Vmax, Vmin
    "bus": np.array([
            [1, 2, 0.0, 0.0, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [2, 1, 300.0, 98.61, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [3, 2, 300.0, 98.61, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [4, 3, 400.0, 131.47, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [5, 2, 0.0, 0.0, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000]
        ]),

    # gen: bus, Pg, Qg, Qmax, Qmin, Vg, mBase, status, Pmax, Pmin
    "gen": np.array([
            [1, 20.0, 0.0, 30.0, -30.0, 1.0, 100.0, 1, 40.0, 0.0],
            [1, 85.0, 0.0, 127.5, -127.5, 1.0, 100.0, 1, 170.0, 0.0],
            [3, 260.0, 0.0, 390.0, -390.0, 1.0, 100.0, 1, 520.0, 0.0],
            [4, 100.0, 0.0, 150.0, -150.0, 1.0, 100.0, 1, 200.0, 0.0],
            [5, 300.0, 0.0, 450.0, -450.0, 1.0, 100.0, 1, 600.0, 0.0]
        ]),

    # gencost: model (2 = polynomial), startup, shutdown, n coefficients, c2, c1, c0
    "gencost": np.array([
            [2, 0.0, 0.0, 3, 0.000000, 14.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 15.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 30.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 40.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 10.000000, 0.000000],
    ]),

    # branch: fbus, tbus, r, x, b, rateA, rateB, rateC, ratio, angle, status, angmin, angmax
    "branch": np.array([
            [1, 2, 0.00281, 0.0281, 0.00712, 400.0, 400.0, 400.0, 0.0, 0.0, 1, -30.0, 30.0],
            [1, 4, 0.00304, 0.0304, 0.00658, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [1, 5, 0.00064, 0.0064, 0.03126, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [2, 3, 0.00108, 0.0108, 0.01852, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [3, 4, 0.00297, 0.0297, 0.00674, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [4, 5, 0.00297, 0.0297, 0.00674, 240.0, 240.0, 240.0, 0.0, 0.0, 1, -30.0, 30.0]
        ])
}

In [4]:
# the same data as labelled tables, for reading
bus_cols = ["bus_i", "type", "Pd", "Qd", "Gs", "Bs", "area", "Vm", "Va", "baseKV", "zone", "Vmax", "Vmin"]
gen_cols = ["bus", "Pg", "Qg", "Qmax", "Qmin", "Vg", "mBase", "status", "Pmax", "Pmin"]
branch_cols = ["fbus", "tbus", "r", "x", "b", "rateA", "rateB", "rateC", "ratio", "angle", "status", "angmin", "angmax"]
cost_cols = ["model", "startup", "shutdown", "n", "c2", "c1", "c0"]

with pd.option_context("display.float_format", "{:g}".format):      # plain numbers (1, not 1.0000)
    print("Bus data");       display(pd.DataFrame(net["bus"], columns=bus_cols))
    print("Generator data"); display(pd.DataFrame(net["gen"], columns=gen_cols))
    print("Branch data");    display(pd.DataFrame(net["branch"], columns=branch_cols))
    print("Generator cost data"); display(pd.DataFrame(net["gencost"], columns=cost_cols))

Bus data


,bus_i,type,Pd,Qd,Gs,Bs,area,Vm,Va,baseKV,zone,Vmax,Vmin
0,1,2,0,0,0,0,1,1,0,230,1,1.1,0.9
1,2,1,300,98.61,0,0,1,1,0,230,1,1.1,0.9
2,3,2,300,98.61,0,0,1,1,0,230,1,1.1,0.9
3,4,3,400,131.47,0,0,1,1,0,230,1,1.1,0.9
4,5,2,0,0,0,0,1,1,0,230,1,1.1,0.9


Generator data


,bus,Pg,Qg,Qmax,Qmin,Vg,mBase,status,Pmax,Pmin
0,1,20,0,30,-30,1,100,1,40,0
1,1,85,0,127.5,-127.5,1,100,1,170,0
2,3,260,0,390,-390,1,100,1,520,0
3,4,100,0,150,-150,1,100,1,200,0
4,5,300,0,450,-450,1,100,1,600,0


Branch data


,fbus,tbus,r,x,b,rateA,rateB,rateC,ratio,angle,status,angmin,angmax
0,1,2,0.00281,0.0281,0.00712,400,400,400,0,0,1,-30,30
1,1,4,0.00304,0.0304,0.00658,426,426,426,0,0,1,-30,30
2,1,5,0.00064,0.0064,0.03126,426,426,426,0,0,1,-30,30
3,2,3,0.00108,0.0108,0.01852,426,426,426,0,0,1,-30,30
4,3,4,0.00297,0.0297,0.00674,426,426,426,0,0,1,-30,30
5,4,5,0.00297,0.0297,0.00674,240,240,240,0,0,1,-30,30


Generator cost data


,model,startup,shutdown,n,c2,c1,c0
0,2,0,0,3,0,14,0
1,2,0,0,3,0,15,0
2,2,0,0,3,0,30,0
3,2,0,0,3,0,40,0
4,2,0,0,3,0,10,0


## Run AC-OPF
`runopf` returns the case dictionary with the solution filled in (generator output, bus voltages, line flows and the dual variables), plus `success` and the objective value `f`. The default options (`PF_DC=0`) select the full AC formulation. `VERBOSE=0` and `OUT_ALL=0` silence the printed report.

AC-OPF is non-convex, so the interior-point solver returns a **locally** optimal point. For this case it is the same point the Pyomo/IPOPT model reaches from a flat start.

In [5]:
option = ppoption(VERBOSE=0, OUT_ALL=0)
results = runopf(net, option)

print(f"Converged: {results['success']}")
print(f"Total cost: ${results['f']:,.2f}/h")

Converged: True
Total cost: $17,551.89/h


## Results

### 1. Generator dispatch

In [6]:
gen_df = pd.DataFrame({
    "Gen": np.arange(1, len(results["gen"]) + 1),
    "Bus": results["gen"][:, GEN_BUS].astype(int),
    "Cost ($/MWh)": results["gencost"][:, 5],                      # c1 (linear cost)
    "Pg (MW)": results["gen"][:, PG],
    "Pmax (MW)": results["gen"][:, PMAX],
    "Qg (MVAr)": results["gen"][:, QG],
    "Qmin (MVAr)": results["gen"][:, QMIN],
    "Qmax (MVAr)": results["gen"][:, QMAX],
})
gen_df["Operating cost ($/h)"] = gen_df["Pg (MW)"] * gen_df["Cost ($/MWh)"]
gen_df

,Gen,Bus,Cost ($/MWh),Pg (MW),Pmax (MW),Qg (MVAr),Qmin (MVAr),Qmax (MVAr),Operating cost ($/h)
0,1,1,14.0000,40.0000,40.0000,29.9998,-30.0000,30.0000,559.9996
1,2,1,15.0000,170.0000,170.0000,127.4998,-127.5000,127.5000,"2,549.9994"
2,3,3,30.0000,324.4981,520.0000,389.9993,-390.0000,390.0000,"9,734.9444"
3,4,4,40.0000,0.0003,200.0000,-10.8018,-150.0000,150.0000,0.0105
4,5,5,10.0000,470.6937,600.0000,-165.0388,-450.0000,450.0000,"4,706.9375"


Gens 1 and 2 run at their active-power maximums, Gen 3 is partly loaded, Gen 5 is held below its 600 MW maximum and Gen 4 stays off (a few kW at most, which is solver tolerance). Gens 1, 2 and 3 all sit at their **reactive** maximums, while Gens 4 and 5 absorb reactive power.

### 2. Bus angles and LMPs

In [7]:
bus_df = pd.DataFrame({
    "Bus": results["bus"][:, BUS_I].astype(int),
    "V (p.u.)": results["bus"][:, VM],
    "theta (deg)": results["bus"][:, VA],
    "LMP P ($/MWh)": results["bus"][:, LAM_P],
    "LMP Q ($/MVAr·h)": results["bus"][:, LAM_Q],
})
bus_df

,Bus,V (p.u.),theta (deg),LMP P ($/MWh),LMP Q ($/MVAr·h)
0,1,1.0776,2.8038,16.9351,0.3570
1,2,1.0841,-0.7346,26.5499,0.3674
2,3,1.1000,-0.5597,30.0000,0.1051
3,4,1.0641,0.0000,39.7121,-0.0000
4,5,1.0691,3.5904,10.0000,-0.0000


`LAM_P` and `LAM_Q` are the multipliers on each bus's active and reactive power balance. `LAM_P` is the **LMP** in \$/MWh. `LAM_Q` is the price of reactive power in \$/MVAr·h. Bus 4 is the reference bus (angle 0), and Bus 3 sits at the 1.10 p.u. voltage upper bound.

### 3. Line flows and shadow prices 

In [8]:
br = results["branch"]
branch_df = pd.DataFrame({
    "Branch": np.arange(1, len(br) + 1),
    "From": br[:, F_BUS].astype(int),
    "To": br[:, T_BUS].astype(int),
    "Pf (MW)": br[:, PF],
    "Qf (MVAr)": br[:, QF],
    "Pt (MW)": br[:, PT],
    "Qt (MVAr)": br[:, QT],
})
branch_df["Sf (MVA)"] = np.hypot(branch_df["Pf (MW)"], branch_df["Qf (MVAr)"])
branch_df["St (MVA)"] = np.hypot(branch_df["Pt (MW)"], branch_df["Qt (MVAr)"])
branch_df["Limit (MVA)"] = br[:, RATE_A]
branch_df["Loading (%)"] = branch_df[["Sf (MVA)", "St (MVA)"]].max(axis=1) / branch_df["Limit (MVA)"] * 100
branch_df["Loss (MW)"] = branch_df["Pf (MW)"] + branch_df["Pt (MW)"]

# shadow price of the thermal limit ($/h per MVA): one multiplier per line end, at most one is nonzero
branch_df["Shadow price ($/h per MVA)"] = br[:, MU_SF] + br[:, MU_ST]

print(f"Total transmission losses: {branch_df['Loss (MW)'].sum():.2f} MW")
branch_df

Total transmission losses: 5.19 MW


,Branch,From,To,Pf (MW),Qf (MVAr),Pt (MW),Qt (MVAr),Sf (MVA),St (MVA),Limit (MVA),Loading (%),Loss (MW),Shadow price ($/h per MVA)
0,1,1,2,252.3778,-42.4497,-250.7938,57.4582,255.9229,257.2916,400.0000,64.3229,1.5840,0.0000
1,2,1,4,187.8686,33.1318,-186.9153,-24.3528,190.7678,188.4950,426.0000,44.7812,0.9534,0.0000
2,3,1,5,-230.2465,166.8175,230.6954,-165.9300,284.3264,284.1709,426.0000,66.7433,0.4489,0.0000
3,4,2,3,-49.2062,-156.0682,49.4492,156.2894,163.6415,163.9256,426.0000,38.4802,0.2430,0.0000
4,5,3,4,-24.9511,135.0999,25.4171,-131.2293,137.3846,133.6681,426.0000,32.2499,0.4660,0.0000
5,6,4,5,-238.5015,13.3104,239.9983,0.8912,238.8726,240.0000,240.0000,100.0000,1.4968,61.3108


Flows are measured at each end of a line, so the active-power losses are `Pf + Pt` (the two ends do not match because some power is lost in the line resistance). `MU_SF` and `MU_ST` are the multipliers on the limit at the from and to end of a line. OFor this case, only one end of Branch 6 has a nonzero thermal-limit multiplier. The line's marginal value is obtained by combining the multipliers from both ends and converting them to a value per MVA of rating.

## Reference Values for the Pyomo Notebook

The cell below prints the arrays in a form that can be pasted into the Pyomo notebook for the comparison.

In [9]:
fmt = lambda a: np.array2string(np.asarray(a, dtype=float), precision=8, separator=", ")

print("# AC-OPF (PYPOWER)")
print("pp_cost  =", results["f"])
print("pp_pg    =", fmt(results["gen"][:, PG]))
print("pp_qg    =", fmt(results["gen"][:, QG]))
print("pp_vm    =", fmt(results["bus"][:, VM]))
print("pp_va    =", fmt(results["bus"][:, VA]))
print("pp_lmp_p =", fmt(results["bus"][:, LAM_P]))
print("pp_lmp_q =", fmt(results["bus"][:, LAM_Q]))
print("pp_mu    =", fmt(results["branch"][:, MU_SF] + results["branch"][:, MU_ST]))

# AC-OPF (PYPOWER)
pp_cost  = 17551.89152713344
pp_pg    = [3.99999742e+01, 1.69999961e+02, 3.24498148e+02, 2.63520422e-04,
 4.70693749e+02]
pp_qg    = [  29.99978758,  127.49978762,  389.99927791,  -10.8017526 , -165.03879745]
pp_vm    = [1.07761726, 1.08406428, 1.09999952, 1.06413689, 1.0690703 ]
pp_va    = [ 2.80378382, -0.7346469 , -0.5597285 ,  0.        ,  3.59035014]
pp_lmp_p = [16.93508208, 26.54990768, 30.00000015, 39.71208642, 10.00000043]
pp_lmp_q = [ 3.57041258e-01,  3.67385756e-01,  1.05113532e-01, -7.34876874e-08,
 -1.42859934e-07]
pp_mu    = [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
 61.31083517]
